# PAI最終課題：小型LLMによる長期タスク分解はVLAの実行を助けるか？
## Notebook 2：LLMによるタスク分解 × SmolVLA実行、ベースラインとの比較

Notebook 1と**同じDriveフォルダ（`pai_final/`）**を使い、`results.jsonl` に条件を追記していきます。
ベースラインの結果はそのまま残るので、同じ初期状態のエピソード同士で比較できます。

| 条件 | SmolVLAに渡す指示 |
|---|---|
| `baseline`（Notebook 1で実施済み） | LIBERO-10の長い指示をそのまま |
| `llm_decomp`（本ノートブック） | 小型LLM（Qwen2.5-1.5B-Instruct）が分解したサブタスクを順に |
| `skill_grounded`（改良版） | LLM分解の各サブタスクを、学習済みスキル（指示文）のうち最も近いものにLLMが置き換えて順に |

**セッションが生きている場合**は、セクション11以降だけを実行すれば続きから進められます。

**ランタイム**：T4 GPU。新しいセッションの場合はセクション1〜5（環境構築）から実行してください。

## 1. GPU確認

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python --version

## 2. インストール
再現性のため、LeRobotは動作確認したコミットに固定しています。

`lerobot[libero]` をそのまま入れると、依存パッケージ `egl_probe` / `hf-egl-probe` のビルドがColab上で失敗します。
この2つ（と、それを要求する `robomimic`）はLIBEROの学習用コード（`libero.lifelong`）でしか使われず、
シミュレーション環境の実行には不要なので、**LIBERO本体を依存なしで入れ、環境の実行に必要な依存だけを追加**します。

In [ ]:
LEROBOT_COMMIT = "3f797de5952432284a7c77ad2760218cb74e2f26"  # 2026-10-30時点のmain

# (1) LeRobot本体＋SmolVLA（libero extraは使わない）
!pip -q install "lerobot[smolvla,dataset,scipy-dep] @ git+https://github.com/huggingface/lerobot.git@{LEROBOT_COMMIT}" imageio imageio-ffmpeg
# (2) LIBERO本体は依存なしでインストール（egl_probe / robomimic を回避）
!pip -q install --no-deps "hf-libero>=0.1.4,<0.2.0"
# (3) LIBEROの環境実行に必要な依存だけを追加
!pip -q install "robosuite==1.4.0" "bddl==1.0.1" "mujoco>=3.0.0,<3.9.0" easydict h5py termcolor cloudpickle future matplotlib "gymnasium>=0.29.0"

!pip list 2>/dev/null | grep -i -E "^(lerobot|hf-libero|robosuite|bddl|mujoco|torch|transformers|numpy) "

## 3. LIBEROの設定ファイル作成・描画設定
LIBEROは初回import時に対話入力（Y/N）を求めるため、設定ファイルを先に作成しておきます。
描画はヘッドレス環境向けにEGLを使います。

In [ ]:
import os, importlib.util, pathlib, yaml

os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

spec = importlib.util.find_spec("libero")
libero_root = pathlib.Path(list(spec.submodule_search_locations)[0]) / "libero"
cfg_dir = pathlib.Path(os.environ.get("LIBERO_CONFIG_PATH", pathlib.Path.home() / ".libero"))
cfg_dir.mkdir(parents=True, exist_ok=True)
paths = {
    "benchmark_root": str(libero_root),
    "bddl_files": str(libero_root / "bddl_files"),
    "init_states": str(libero_root / "init_files"),
    "datasets": str(libero_root.parent / "datasets"),
    "assets": str(libero_root / "assets"),
}
with open(cfg_dir / "config.yaml", "w") as f:
    yaml.dump(paths, f)
print(open(cfg_dir / "config.yaml").read())

from libero.libero import benchmark  # 対話入力なしでimportできればOK
print("LIBERO suites:", sorted(benchmark.get_benchmark_dict().keys()))

## 4. Google Driveのマウントと保存先
無料枠ではGPUが途中で切断されることがあるため、結果は1エピソードごとにDriveへ保存します。
Driveを使わない場合は `USE_DRIVE = False` にしてください（その場合、切断時に結果が消えます）。

In [ ]:
USE_DRIVE = True

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_DIR = pathlib.Path("/content/drive/MyDrive/pai_final")
else:
    OUT_DIR = pathlib.Path("/content/pai_final")

(OUT_DIR / "videos").mkdir(parents=True, exist_ok=True)
RESULTS_PATH = OUT_DIR / "results.jsonl"
print("保存先:", OUT_DIR)

## 5. 実験設定
- **ポリシー**：LeRobot公式CIでLIBERO評価に使われている設定（カメラ名 `camera1`/`camera2`、`empty_cameras=1`）をそのまま使います。
  スモークテストで成功しない場合は、`POLICY_PRESET = "hfvla"` に切り替えて再度セル5〜6を実行してください。
- **タスク**：LIBERO-10のうち「2つの物体を順に操作する」タスク4つ

In [ ]:
POLICY_PRESET = "lerobot_ci"   # "lerobot_ci" または "hfvla"

PRESETS = {
    "lerobot_ci": dict(
        path="lerobot/smolvla_libero",
        camera_name_mapping={"agentview_image": "camera1", "robot0_eye_in_hand_image": "camera2"},
        policy_overrides=["--empty_cameras=1"],
    ),
    "hfvla": dict(
        path="HuggingFaceVLA/smolvla_libero",
        camera_name_mapping=None,  # 既定（image / image2）
        policy_overrides=[],
    ),
}
P = PRESETS[POLICY_PRESET]

SUITE = "libero_10"
TASK_IDS = [0, 1, 4, 7]   # すべて「2つの物体」を扱うタスク
N_EPISODES = 5            # 1タスクあたりのエピソード数（初期状態0〜4を固定で使用）
SEED = 1000
DEVICE = "cuda"
print(P)

## 6. 小型LLMによるタスク分解
- **モデル**：Qwen2.5-1.5B-Instruct（fp16で約3GB、T4で十分動きます）
- **設計**：分解はタスクごとに1回だけ実行してDriveにキャッシュします。評価中はLLMを使わないので、GPUメモリを解放してからSmolVLAを読み込みます。
- **公平性**：プロンプトの例（one-shot）には、評価対象の4タスクとは**別の指示**を使っています。評価タスクの正解分解はプロンプトに入れていません。
- **デコード**：貪欲法（`do_sample=False`）で決定的に生成し、生の出力もそのまま保存します。JSONとして読めない場合は元の指示にフォールバックし、その旨を記録します。

In [ ]:
import json, re, torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from libero.libero import benchmark as _bm

LLM_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
DECOMP_PATH = OUT_DIR / "decompositions.json"

SYSTEM_PROMPT = (
    "You are a task planner for a single-arm tabletop robot. "
    "The robot can only execute short, single-step manipulation instructions, such as "
    "'put the <object> in the <container>', 'put the <object> on the <target>', "
    "'pick up the <object> and place it <location>', 'open the <thing>', 'close the <thing>', "
    "'turn on the <appliance>'. "
    "Split the user's instruction into the minimal ordered list of single-step instructions "
    "that the robot should execute one after another. "
    "Keep object and place names exactly as written in the user's instruction. "
    "Output only a JSON array of strings, with no explanation."
)
# one-shot例：評価対象（LIBERO-10のtask 0, 1, 4, 7）とは別の指示
FEWSHOT = [
    {"role": "user", "content": "open the top drawer and put the bowl inside"},
    {"role": "assistant", "content": '["open the top drawer of the cabinet", "put the bowl in the top drawer of the cabinet"]'},
]

_suite_tmp = _bm.get_benchmark_dict()[SUITE]()
TASK_TEXT = {tid: _suite_tmp.get_task(tid).language for tid in TASK_IDS}

def parse_subtasks(text: str):
    m = re.search(r"\[.*\]", text, flags=re.S)
    if not m:
        return None
    try:
        arr = json.loads(m.group(0))
    except json.JSONDecodeError:
        return None
    arr = [s.strip() for s in arr if isinstance(s, str) and s.strip()]
    return arr or None

decomp = json.load(open(DECOMP_PATH)) if DECOMP_PATH.exists() else {}
todo = [tid for tid in TASK_IDS if str(tid) not in decomp]

if todo:
    tok = AutoTokenizer.from_pretrained(LLM_NAME)
    llm = AutoModelForCausalLM.from_pretrained(LLM_NAME, dtype=torch.float16, device_map="cuda")
    llm.eval()
    for tid in todo:
        msgs = [{"role": "system", "content": SYSTEM_PROMPT}, *FEWSHOT,
                {"role": "user", "content": TASK_TEXT[tid]}]
        prompt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        inputs = tok(prompt, return_tensors="pt").to(llm.device)
        with torch.no_grad():
            out = llm.generate(**inputs, max_new_tokens=128, do_sample=False,
                               pad_token_id=tok.eos_token_id)
        raw = tok.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        subtasks = parse_subtasks(raw)
        decomp[str(tid)] = {
            "task": TASK_TEXT[tid], "raw_output": raw,
            "subtasks": subtasks or [TASK_TEXT[tid]], "parse_ok": subtasks is not None,
            "llm": LLM_NAME,
        }
    json.dump(decomp, open(DECOMP_PATH, "w"), ensure_ascii=False, indent=1)
    json.dump({"llm": LLM_NAME, "system_prompt": SYSTEM_PROMPT, "fewshot": FEWSHOT},
              open(OUT_DIR / "llm_prompt.json", "w"), ensure_ascii=False, indent=1)
    del llm; torch.cuda.empty_cache()

for tid in TASK_IDS:
    d = decomp[str(tid)]
    print(f"[{tid}] {d['task']}")
    for k, s in enumerate(d["subtasks"], 1):
        print(f"    {k}. {s}")
    if not d["parse_ok"]:
        print("    ※ JSONとして読めなかったため元の指示にフォールバック。raw:", d["raw_output"])
print(f"\nGPU memory after LLM: {torch.cuda.memory_allocated()/1e9:.2f} GB")

分解結果がおかしい場合（例：物体名が変わっている、3つ以上に分かれている）も、**手で直さずにそのまま評価**します。
「小型LLMの出力をそのまま使ったらどうなるか」が実験の主題なので、問題があればそれ自体を考察に使います。

## 7. ポリシーと前処理パイプラインの読み込み
公式 `lerobot-eval` と同じ手順（`make_policy` → `make_pre_post_processors` → `make_env_pre_post_processors`）で組み立てます。

In [ ]:
from pathlib import Path
import numpy as np, torch, gymnasium as gym

from lerobot.configs import PreTrainedConfig
from lerobot.envs import make_env_pre_post_processors, preprocess_observation
from lerobot.envs.configs import LiberoEnv as LiberoEnvConfig
from lerobot.envs import libero as libero_mod
from lerobot.policies import make_policy, make_pre_post_processors
from lerobot.processor import policy_output_to_transition, transition_to_policy_action
from lerobot.utils.constants import ACTION

torch.manual_seed(SEED); np.random.seed(SEED)

env_cfg = LiberoEnvConfig(task=SUITE, task_ids=TASK_IDS, camera_name_mapping=P["camera_name_mapping"])

policy_cfg = PreTrainedConfig.from_pretrained(
    P["path"], cli_overrides=P["policy_overrides"] + [f"--device={DEVICE}"]
)
policy_cfg.pretrained_path = Path(P["path"])

policy = make_policy(cfg=policy_cfg, env_cfg=env_cfg)
policy.eval()

preprocessor, postprocessor = make_pre_post_processors(
    policy_cfg=policy_cfg,
    pretrained_path=policy_cfg.pretrained_path,
    preprocessor_overrides={"device_processor": {"device": str(policy.config.device)}},
)
env_preprocessor, env_postprocessor = make_env_pre_post_processors(env_cfg=env_cfg, policy_cfg=policy_cfg)

print("input features :", list(policy.config.input_features.keys()))
print("n_action_steps :", policy.config.n_action_steps, "/ chunk_size:", policy.config.chunk_size)
print(f"GPU memory     : {torch.cuda.memory_allocated()/1e9:.2f} GB")

## 8. 評価ループの定義
`run_episode(task_id, instructions, episode_index)` は、**ベースライン（指示1つ）とLLM分解（指示が複数）の両方で共通**に使います。

- **サブタスクの切り替え規則**：各ステップでLIBEROのゴール述語（BDDLの `:goal` に並ぶ条件）を個別に評価し、満たされた述語の数 *k* に応じて *k* 番目の指示を渡します。
  これはシミュレータの内部状態を使う「オラクル判定」なので、レポートでは限界として明記します。
- 指示が切り替わったら `policy.reset()` で行動チャンクのキューを破棄し、新しい指示で推論し直します。
- **初期状態の固定**：`episode_index` ごとにLIBEROの初期状態を固定するため、条件間で同じ初期配置を比較できます。
- 部分達成（満たした述語の数）も記録し、「どこまで進んだか」も分析できるようにします。

In [ ]:
import time, json, imageio
from PIL import Image, ImageDraw, ImageFont

_SUITE_OBJ = libero_mod._get_suite(SUITE)
_GYM_KW = {k: v for k, v in env_cfg.gym_kwargs.items() if k != "task_ids"}

def make_vec_env(task_id: int, episode_index: int):
    def _fn():
        return libero_mod.LiberoEnv(
            task_suite=_SUITE_OBJ, task_id=task_id, task_suite_name=SUITE,
            camera_name=env_cfg.camera_name, init_states=env_cfg.init_states,
            episode_length=env_cfg.episode_length, episode_index=episode_index, n_envs=1,
            control_mode=env_cfg.control_mode, camera_name_mapping=env_cfg.camera_name_mapping,
            **_GYM_KW,
        )
    return gym.vector.SyncVectorEnv([_fn], autoreset_mode=gym.vector.AutoresetMode.NEXT_STEP)

def task_language(task_id: int) -> str:
    return _SUITE_OBJ.get_task(task_id).language

try:
    _FONT = ImageFont.load_default(size=15)
except TypeError:
    _FONT = ImageFont.load_default()

def annotate(frame: np.ndarray, lines: list[str], scale: int = 2) -> np.ndarray:
    img = Image.fromarray(frame).resize((frame.shape[1] * scale, frame.shape[0] * scale), Image.BICUBIC)
    draw = ImageDraw.Draw(img)
    pad, lh = 6, 19
    draw.rectangle([0, 0, img.width, pad * 2 + lh * len(lines)], fill=(0, 0, 0))
    for i, t in enumerate(lines):
        draw.text((pad, pad + i * lh), t, fill=(255, 255, 255), font=_FONT)
    return np.asarray(img)

@torch.no_grad()
def run_episode(task_id: int, instructions: list[str], episode_index: int,
                condition: str = "baseline", video_path: str | None = None) -> dict:
    t0 = time.time()
    venv = make_vec_env(task_id, episode_index)
    obs, _ = venv.reset(seed=SEED + episode_index)
    lenv = venv.envs[0]
    inner = lenv._env.env                          # LIBEROの問題ドメイン（BDDL）
    goals = inner.parsed_problem["goal_state"]     # 例: [['in','alphabet_soup_1','basket_1_contain_region'], ...]
    max_steps = lenv._max_episode_steps

    policy.reset()
    cur, switches, frames = 0, [], []
    best_sat, success, steps = 0, False, 0

    def eval_goals():
        return [bool(inner._eval_predicate(g)) for g in goals]

    for step in range(max_steps):
        sat = eval_goals()
        n_sat = sum(sat); best_sat = max(best_sat, n_sat)
        target = min(n_sat, len(instructions) - 1)
        if target != cur:
            cur = target
            policy.reset()
            switches.append({"step": step, "to": cur})

        o = preprocess_observation(obs)
        o["task"] = [instructions[cur]]
        o = env_preprocessor(o)
        o = preprocessor(o)
        with torch.inference_mode():
            act = transition_to_policy_action(policy_output_to_transition(policy.select_action(o)))
        act = postprocessor(act)
        act = env_postprocessor({ACTION: act})[ACTION]

        obs, _, terminated, truncated, info = venv.step(act.to("cpu").numpy())
        steps = step + 1

        if video_path is not None:
            frames.append(annotate(lenv.render(), [
                f"[{condition}] task {task_id} / ep {episode_index} / step {steps}",
                f"instruction {cur + 1}/{len(instructions)}: {instructions[cur]}",
                f"goals satisfied: {n_sat}/{len(goals)}",
            ]))

        if bool(np.asarray(terminated)[0]) or bool(np.asarray(truncated)[0]):
            success = bool(np.asarray(info.get("is_success", [False]))[0])
            break

    final_sat = eval_goals()
    success = success or all(final_sat)
    venv.close()

    if video_path is not None and frames:
        imageio.mimsave(video_path, frames, fps=env_cfg.fps)

    return {
        "task_id": task_id, "task": task_language(task_id), "condition": condition,
        "episode": episode_index, "instructions": instructions,
        "success": success, "steps": steps, "max_steps": max_steps,
        "n_goals": len(goals), "best_goals_satisfied": max(best_sat, sum(final_sat)),
        "final_goals_satisfied": sum(final_sat), "switches": switches,
        "wall_sec": round(time.time() - t0, 1), "policy": P["path"],
    }

## 9. 対象タスクとゴール述語の確認
LLM分解のプロンプト設計（Notebook 2）でも使うので、各タスクの指示文と述語を確認しておきます。

In [ ]:
for tid in TASK_IDS:
    venv = make_vec_env(tid, 0); venv.reset(seed=SEED)
    goals = venv.envs[0]._env.env.parsed_problem["goal_state"]
    print(f"[{tid}] {task_language(tid)}")
    for g in goals:
        print("      -", g)
    venv.close()

## 10. LLM分解条件の本番評価
Notebook 1のベースラインと同じ初期状態（エピソード0〜4）で実行します。途中で切断されても、再実行すれば続きから再開します。

In [ ]:
def load_done(path) -> set:
    done = set()
    if Path(path).exists():
        for line in open(path):
            r = json.loads(line)
            done.add((r["condition"], r["task_id"], r["episode"]))
    return done

def run_condition(condition: str, instr_fn, save_video_eps=(0,)):
    done = load_done(RESULTS_PATH)
    for tid in TASK_IDS:
        for ep in range(N_EPISODES):
            if (condition, tid, ep) in done:
                continue
            vp = str(OUT_DIR / "videos" / f"{condition}_t{tid}_e{ep}.mp4") if ep in save_video_eps else None
            r = run_episode(tid, instr_fn(tid), ep, condition=condition, video_path=vp)
            with open(RESULTS_PATH, "a") as f:
                f.write(json.dumps(r, ensure_ascii=False) + "\n")
            print(f"[{condition}] task {tid} ep {ep}: success={r['success']} "
                  f"goals={r['best_goals_satisfied']}/{r['n_goals']} steps={r['steps']} ({r['wall_sec']}s)")

run_condition("llm_decomp", lambda tid: decomp[str(tid)]["subtasks"])

## 11. LLMの分解品質の確認（人間の分解との一致）
LLMの分解が人間の分解と一致していれば、`llm_decomp` の失敗は「計画の誤り」ではなく
「分解後の指示をVLAが実行できない」ことに起因すると言えます。人間の分解を別条件として再評価する代わりに、ここで一致を確認します。

In [ ]:
MANUAL = {
    0: ["put the alphabet soup in the basket", "put the tomato sauce in the basket"],
    1: ["put the cream cheese box in the basket", "put the butter in the basket"],
    4: ["put the white mug on the left plate", "put the yellow and white mug on the right plate"],
    7: ["put the alphabet soup in the basket", "put the cream cheese box in the basket"],
}
for tid in TASK_IDS:
    same = decomp[str(tid)]["subtasks"] == MANUAL[tid]
    print(f"[{tid}] LLM分解 == 人間の分解: {same}")

## 12. 改良版：スキルライブラリに接地した分解（`skill_grounded`）
**着想**：`llm_decomp` では分解自体は正しいのに、SmolVLAが1つ目のサブタスクすら実行できませんでした。
smolvla_liberoは4つのスイートの指示文で学習されており、単発の指示はLIBERO-Object（`pick up the X and place it in the basket`）や
LIBERO-Goal、LIBERO-Spatialの決まった言い回しでしか見ていません。

そこでSayCanのように、**ロボットが学習済みの指示文（スキルライブラリ）を候補として与え、LLMに各サブタスクに最も近いスキルを選ばせます**。
- スキルライブラリ：LIBERO-Object / Goal / Spatial の学習用指示文30個（評価対象のLIBERO-10の指示は含めない）
- 小型LLMでも安定するよう、自由記述ではなく**番号を選ぶ多肢選択**にします（0＝該当なし→元のサブタスクのまま）
- 選択結果もDriveに保存し、レポートに載せます

In [ ]:
import gc
SKILL_SUITES = ["libero_object", "libero_goal", "libero_spatial"]
_bd = _bm.get_benchmark_dict()
SKILLS = []
for s in SKILL_SUITES:
    suite_s = _bd[s]()
    for i in range(len(suite_s.tasks)):
        lang = suite_s.get_task(i).language
        if lang not in SKILLS:
            SKILLS.append(lang)
print(f"スキル数: {len(SKILLS)}")

GROUND_PATH = OUT_DIR / "grounded_decompositions.json"
SELECT_PROMPT = (
    "A robot can only execute the following learned skills, written exactly as listed.\n"
    "{skills}\n\n"
    "Which skill has the same meaning as this step: \"{step}\"?\n"
    "The object and the destination must match. If no skill matches, answer 0.\n"
    "Answer with the number only."
)
skill_list_text = "0. (none of these)\n" + "\n".join(f"{i}. {s}" for i, s in enumerate(SKILLS, 1))

grounded = json.load(open(GROUND_PATH)) if GROUND_PATH.exists() else {}
todo = [tid for tid in TASK_IDS if str(tid) not in grounded]
if todo:
    tok = AutoTokenizer.from_pretrained(LLM_NAME)
    llm = AutoModelForCausalLM.from_pretrained(LLM_NAME, dtype=torch.float16, device_map="cuda").eval()
    for tid in todo:
        steps_out = []
        for step in decomp[str(tid)]["subtasks"]:
            msgs = [{"role": "user", "content": SELECT_PROMPT.format(skills=skill_list_text, step=step)}]
            prompt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
            inputs = tok(prompt, return_tensors="pt").to(llm.device)
            with torch.no_grad():
                out = llm.generate(**inputs, max_new_tokens=8, do_sample=False, pad_token_id=tok.eos_token_id)
            raw = tok.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True)
            m = re.search(r"\d+", raw)
            idx = int(m.group(0)) if m else 0
            chosen = SKILLS[idx - 1] if 1 <= idx <= len(SKILLS) else None
            steps_out.append({"step": step, "raw": raw, "index": idx,
                              "grounded": chosen or step, "matched": chosen is not None})
        grounded[str(tid)] = {"task": TASK_TEXT[tid], "steps": steps_out,
                              "subtasks": [s["grounded"] for s in steps_out]}
    json.dump(grounded, open(GROUND_PATH, "w"), ensure_ascii=False, indent=1)
    del llm; gc.collect(); torch.cuda.empty_cache()

for tid in TASK_IDS:
    print(f"[{tid}] {TASK_TEXT[tid]}")
    for s in grounded[str(tid)]["steps"]:
        mark = "→" if s["matched"] else "×(該当なし・元のまま)"
        print(f"    {s['step']}  {mark}  {s['grounded'] if s['matched'] else ''}")

選択結果がおかしくても（例：違う物体のスキルを選んだ）、手で直さずにそのまま評価します。

In [ ]:
run_condition("skill_grounded", lambda tid: grounded[str(tid)]["subtasks"])

## 13. 比較・集計
- 条件×タスクの成功率と部分達成率（満たしたゴール述語の割合）
- **同じ初期状態でのペア比較**：ベースラインで失敗→分解で成功（改善）、その逆（悪化）のエピソード数
- 図はレポート・動画にそのまま使えるようDriveに保存します

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_json(RESULTS_PATH, lines=True)
df = df[df["condition"] != "manual_test"].copy()
df["progress"] = df["best_goals_satisfied"] / df["n_goals"]
order = [c for c in ["baseline", "llm_decomp", "skill_grounded"] if c in set(df["condition"])]

overall = (df.groupby("condition").agg(n=("success", "size"), success_rate=("success", "mean"),
                                        mean_progress=("progress", "mean"), mean_steps=("steps", "mean"))
             .reindex(order))
print("=== 全体 ===")
display(overall.style.format({"success_rate": "{:.0%}", "mean_progress": "{:.0%}", "mean_steps": "{:.0f}"}))

by_task = (df.groupby(["task_id", "condition"]).agg(n=("success", "size"), success_rate=("success", "mean"),
                                                    mean_progress=("progress", "mean"), mean_steps=("steps", "mean"))
             .reset_index())
print("=== タスク別 ===")
display(by_task.style.format({"success_rate": "{:.0%}", "mean_progress": "{:.0%}", "mean_steps": "{:.0f}"}))
by_task.to_csv(OUT_DIR / "summary_by_task.csv", index=False)
overall.to_csv(OUT_DIR / "summary_overall.csv")

# 同じ初期状態でのペア比較（baseline vs 各分解条件）
base = df[df["condition"] == "baseline"].set_index(["task_id", "episode"])
rows = []
for cond in order[1:]:
    other = df[df["condition"] == cond].set_index(["task_id", "episode"])
    j = base[["success", "best_goals_satisfied"]].join(other[["success", "best_goals_satisfied"]],
                                                       lsuffix="_base", rsuffix="_dec", how="inner")
    rows.append({"condition": cond, "paired_episodes": len(j),
                 "改善 (失敗→成功)": int((~j.success_base & j.success_dec).sum()),
                 "悪化 (成功→失敗)": int((j.success_base & ~j.success_dec).sum()),
                 "両方成功": int((j.success_base & j.success_dec).sum()),
                 "両方失敗": int((~j.success_base & ~j.success_dec).sum())})
paired = pd.DataFrame(rows)
print("=== ペア比較（同じ初期状態） ===")
display(paired)
paired.to_csv(OUT_DIR / "paired_comparison.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
for ax, col, title in [(axes[0], "success_rate", "Success rate"), (axes[1], "mean_progress", "Goal progress")]:
    by_task.pivot(index="task_id", columns="condition", values=col)[order].plot(kind="bar", ax=ax, ylim=(0, 1), rot=0)
    ax.set_title(title); ax.set_xlabel("LIBERO-10 task id"); ax.set_ylabel("")
plt.suptitle("SmolVLA on LIBERO-10: long instruction vs. LLM-decomposed subtasks")
plt.tight_layout(); plt.savefig(OUT_DIR / "comparison.png", dpi=150); plt.show()

## 14. 失敗パターンの確認（考察用）
分解条件で、サブタスクの切り替えが起きたか（＝1つ目を達成できたか）と、その後どこで止まったかを一覧にします。

In [ ]:
cols = ["condition", "task_id", "episode", "success", "best_goals_satisfied", "n_goals", "steps", "switches"]
display(df[df["condition"] != "baseline"][cols].sort_values(["condition", "task_id", "episode"]).reset_index(drop=True))
print("保存済み動画:", sorted(p.name for p in (OUT_DIR / "videos").glob("*.mp4")))